In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import argparse
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import scipy.sparse

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score


# --------------------------------------------------------------
# Utility: device
# --------------------------------------------------------------
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --------------------------------------------------------------
# Data loading
# --------------------------------------------------------------
def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    """
    Loads a clean CSV for a dataset.
    Assumes path: base_dir/clean/{dataset_name}_clean.csv
    and columns: ['text', 'label', 'split'] where split in {train,val,test}.
    """
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Clean CSV not found at: {csv_path}")

    df = pd.read_csv(csv_path)
    expected_cols = {"text", "label", "split"}
    if not expected_cols.issubset(set(df.columns)):
        raise ValueError(
            f"{csv_path} must contain columns {expected_cols}, "
            f"but has {set(df.columns)}"
        )

    train_df = df[df["split"] == "train"]
    val_df = df[df["split"] == "val"]
    test_df = df[df["split"] == "test"]

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError(
            f"{dataset_name}: one of the splits is empty. "
            f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}"
        )

    return df.reset_index(drop=True)


def load_graph_npz(base_dir: str, dataset_name: str, k: int) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Loads a k-NN graph saved as .npz with 'row' and 'col' arrays.
    Expected path: base_dir/graphs/{dataset_name}/knn_k{k}.npz
    Returns: row, col, num_nodes
    """
    graph_path = os.path.join(base_dir, "graphs", dataset_name, f"knn_k{k}.npz")
    if not os.path.exists(graph_path):
        raise FileNotFoundError(f"Graph file not found at: {graph_path}")

    try:
        sparse_matrix = scipy.sparse.load_npz(graph_path)
        row, col = sparse_matrix.nonzero()
        num_nodes = sparse_matrix.shape[0]
        return row, col, num_nodes
    except Exception as e:
        data = np.load(graph_path)
        if "row" in data and "col" in data:
            row = data["row"]
            col = data["col"]
        elif "sources" in data and "targets" in data:
            row = data["sources"]
            col = data["targets"]
        elif "indices" in data and "indptr" in data and "shape" in data:
            sparse_matrix_reconstructed = scipy.sparse.csr_matrix(
                (np.ones(len(data["indices"])), data["indices"], data["indptr"]),
                shape=data["shape"]
            )
            row, col = sparse_matrix_reconstructed.nonzero()
        else:
            raise ValueError(
                f"Graph file {graph_path} does not contain 'row'/'col', "
                f"'sources'/'targets', or sparse matrix components ('indices', 'indptr', 'shape'). "
                f"Keys found: {list(data.keys())}. Original error: {e}"
            )

        num_nodes = int(data["num_nodes"]) if "num_nodes" in data else int(max(row.max(), col.max()) + 1)
        return row, col, num_nodes


# --------------------------------------------------------------
# Transformer encoding
# --------------------------------------------------------------
def encode_texts_for_dataset(
    df: pd.DataFrame,
    tokenizer,
    transformer,
    device,
    max_length: int = 128,
    batch_size: int = 32,
) -> torch.Tensor:
    """
    Encodes all texts in df['text'] into a tensor of shape (N, hidden_dim)
    using a frozen transformer model.
    """
    texts = df["text"].astype(str).tolist()
    all_embeddings = []

    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size : (i + 1) * batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())

    embeddings = torch.cat(all_embeddings, dim=0)
    return embeddings


# --------------------------------------------------------------
# GCN Layer (no torch_geometric, no sparse indices bug)
# --------------------------------------------------------------
class GCNLayer(nn.Module):
    """
    Simple GCN layer implemented with edge_index and dense node features.
    A_hat = D^{-1/2} (A + I) D^{-1/2}
    """

    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.0):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,          # (N, d_in)
        edge_index: torch.Tensor, # (2, E)
        num_nodes: int,
    ) -> torch.Tensor:
        row, col = edge_index
        device = x.device

        self_loop = torch.arange(num_nodes, device=device)
        row = torch.cat([row, self_loop], dim=0)
        col = torch.cat([col, self_loop], dim=0)

        deg = torch.zeros(num_nodes, device=device)
        deg.index_add_(0, row, torch.ones_like(row, dtype=deg.dtype, device=device))

        deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)

        x = self.linear(x)
        x = self.dropout(x)

        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        x_col = x[col] * norm.unsqueeze(-1)

        out = torch.zeros_like(x)
        out.index_add_(0, row, x_col)

        return out


# --------------------------------------------------------------
# Hybrid GCN + Transformer Embedding Model
# --------------------------------------------------------------
class HybridGCNTransformer(nn.Module):
    def __init__(
        self,
        emb_dim: int,
        gcn_hidden: int = 256,
        fusion_hidden: int = 256,
        num_classes: int = 2,
        dropout: float = 0.3,   # CHANGED (Fix #2): was 0.5, reduced to 0.3
    ):
        super().__init__()
        self.gcn1 = GCNLayer(emb_dim, gcn_hidden, dropout=dropout)
        self.gcn2 = GCNLayer(gcn_hidden, gcn_hidden, dropout=dropout)

        fusion_input_dim = emb_dim + gcn_hidden
        self.fc_fuse = nn.Linear(fusion_input_dim, fusion_hidden)
        self.fc_out = nn.Linear(fusion_hidden, num_classes)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
        g = F.relu(self.gcn1(x, edge_index, num_nodes))
        g = F.relu(self.gcn2(g, edge_index, num_nodes))

        fused = torch.cat([x, g], dim=-1)
        fused = F.relu(self.fc_fuse(fused))
        fused = self.dropout(fused)
        logits = self.fc_out(fused)
        return logits


# --------------------------------------------------------------
# Build dataset structures for all four datasets
# --------------------------------------------------------------
def prepare_all_datasets(
    base_dir: str,
    model_name: str,
    max_length: int,
    batch_size: int,
    k: int,
    device,
) -> Dict[str, Dict]:
    """
    For each dataset:
      - load clean CSV
      - encode text with transformer
      - load graph (row, col)
      - build masks for train/val/test
    Returns a dict keyed by dataset name.
    """
    datasets = ["liar", "coaid", "fakenewsnet", "fakehealth"]

    print(f"Using device: {device}")
    print(f"Base data dir: {base_dir}")
    print(f"Transformer: {model_name}, k={k}")
    print("Loading transformer and tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)

    all_data = {}

    for name in datasets:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")

        embeddings = encode_texts_for_dataset(
            df,
            tokenizer,
            transformer,
            device,
            max_length=max_length,
            batch_size=batch_size,
        )  # on CPU

        labels = torch.tensor(df["label"].values, dtype=torch.long)

        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)

        print("  Loading graph...")
        row, col, num_nodes = load_graph_npz(base_dir, name, k)
        if num_nodes != len(df):
            print(
                f"  [Warning] num_nodes in graph ({num_nodes}) != num_rows in df ({len(df)}). "
                "Using df length as num_nodes and clipping indices if needed."
            )
            num_nodes = len(df)
            row = np.clip(row, 0, num_nodes - 1)
            col = np.clip(col, 0, num_nodes - 1)

        edge_index = torch.tensor(
            np.vstack([row, col]),
            dtype=torch.long,
        )  # (2, E)

        all_data[name] = {
            "df": df,
            "emb": embeddings,       # (N, emb_dim) on CPU
            "labels": labels,
            "train_mask": train_mask,
            "val_mask": val_mask,
            "test_mask": test_mask,
            "edge_index": edge_index,
            "num_nodes": num_nodes,
        }

    return all_data


# --------------------------------------------------------------
# NEW (Fix #1): compute inverse-frequency class weights from
# the SOURCE dataset's train_mask labels only.
# --------------------------------------------------------------
def compute_class_weights(y_src: torch.Tensor, train_mask: torch.Tensor, num_classes: int, device) -> torch.Tensor:
    """
    Standard inverse-frequency weighting:
        weight_c = N_train / (num_classes * count_c)
    Computed ONLY from the training split of the current source dataset,
    never from val/test, to avoid leaking label distribution information.
    """
    train_labels = y_src[train_mask]
    counts = torch.bincount(train_labels, minlength=num_classes).float()
    counts = counts.clamp(min=1.0)  # avoid div-by-zero if a class is absent
    n_train = train_labels.size(0)
    weights = n_train / (num_classes * counts)
    return weights.to(device)


# --------------------------------------------------------------
# Training and evaluation for one source dataset
# --------------------------------------------------------------
def train_one_source(
    src_name: str,
    all_data: Dict[str, Dict],
    device,
    epochs: int = 30,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    gcn_hidden: int = 256,
    fusion_hidden: int = 256,
    dropout: float = 0.3,   # CHANGED (Fix #2): was 0.5
) -> Dict[str, Dict[str, float]]:
    """
    Trains the hybrid model on src_name and evaluates on all datasets.
    Returns a dict: {target_name: {"acc": ..., "macro_f1": ...}, ...}
    """
    data_src = all_data[src_name]

    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    train_mask = data_src["train_mask"].to(device)
    val_mask = data_src["val_mask"].to(device)
    edge_index_src = data_src["edge_index"].to(device)
    num_nodes_src = data_src["num_nodes"]

    emb_dim = x_src.size(1)
    model = HybridGCNTransformer(
        emb_dim=emb_dim,
        gcn_hidden=gcn_hidden,
        fusion_hidden=fusion_hidden,
        num_classes=2,
        dropout=dropout,
    ).to(device)

    # DEBUG: confirm this is a fresh, distinctly-initialized model per source
    init_checksum = sum(p.sum().item() for p in model.parameters())
    print(f"  [DEBUG] [{src_name}] freshly-initialized model checksum: {init_checksum:.6f}")

    # NEW (Fix #1): class-weighted loss computed from this source's train labels
    class_weights = compute_class_weights(y_src, train_mask, num_classes=2, device=device)
    print(f"  [DEBUG] [{src_name}] class weights (real, fake order matches label ints): {class_weights.cpu().tolist()}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    print(f"\n=== Training on source dataset: {src_name} ===")
    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x_src, edge_index_src, num_nodes_src)
        loss = criterion(logits[train_mask], y_src[train_mask])
        loss.backward()
        optimizer.step()

        # Fix (from v2): re-run forward pass AFTER the weight update,
        # under eval(), instead of reusing pre-step training logits
        model.eval()
        with torch.no_grad():
            val_logits_full = model(x_src, edge_index_src, num_nodes_src)
            val_logits = val_logits_full[val_mask]
            val_labels = y_src[val_mask]
            val_pred = val_logits.argmax(dim=-1)
            val_acc = accuracy_score(val_labels.cpu().numpy(), val_pred.cpu().numpy())
            val_f1 = f1_score(val_labels.cpu().numpy(), val_pred.cpu().numpy(), average="macro")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            # NEW (Fix #1 diagnostic): also print prediction class balance to catch collapse early
            pred_counts = torch.bincount(val_pred, minlength=2).cpu().tolist()
            print(
                f"  Epoch {epoch:03d} | Loss={loss.item():.4f} "
                f"| Val Acc={val_acc:.4f} | Val F1={val_f1:.4f} "
                f"| Val pred class counts (fake,real)={pred_counts}"
            )

    print(f"  Best Val Macro-F1 for {src_name}: {best_val_f1:.4f}")

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # DEBUG: checksum of the FINAL model actually used for evaluation
    final_checksum = sum(p.sum().item() for p in model.parameters())
    print(f"  [DEBUG] [{src_name}] final loaded-best-model checksum: {final_checksum:.6f}")

    results = {}
    for tgt_name, data_tgt in all_data.items():
        x_tgt = data_tgt["emb"].to(device)
        y_tgt = data_tgt["labels"].to(device)
        test_mask = data_tgt["test_mask"].to(device)
        edge_index_tgt = data_tgt["edge_index"].to(device)
        num_nodes_tgt = data_tgt["num_nodes"]

        model.eval()
        with torch.no_grad():
            logits_tgt = model(x_tgt, edge_index_tgt, num_nodes_tgt)
            logits_tgt = logits_tgt[test_mask]
            labels_tgt = y_tgt[test_mask]
            pred_tgt = logits_tgt.argmax(dim=-1)

            acc = accuracy_score(labels_tgt.cpu().numpy(), pred_tgt.cpu().numpy())
            macro_f1 = f1_score(
                labels_tgt.cpu().numpy(), pred_tgt.cpu().numpy(), average="macro"
            )
            pred_counts_test = torch.bincount(pred_tgt, minlength=2).cpu().tolist()

        print(
            f"  [Train={src_name} \u2192 Test={tgt_name}] "
            f"acc={acc:.4f}, macro-F1={macro_f1:.4f}, pred class counts (fake,real)={pred_counts_test}"
        )

        results[tgt_name] = {"acc": acc, "macro_f1": macro_f1}

    return results


# --------------------------------------------------------------
# Cross-domain loop
# --------------------------------------------------------------
def run_cross_domain_hybrid(
    base_dir: str,
    model_name: str = "roberta-base",
    epochs: int = 30,
    max_length: int = 128,
    batch_size: int = 32,
    k: int = 10,
    dropout: float = 0.3,
    datasets_override=None,
):
    device = get_device()
    all_data = prepare_all_datasets(
        base_dir=base_dir,
        model_name=model_name,
        max_length=max_length,
        batch_size=batch_size,
        k=k,
        device=device,
    )

    datasets = ["liar", "coaid", "fakenewsnet", "fakehealth"]
    source_datasets = datasets_override if datasets_override is not None else datasets

    macro_f1_table = pd.DataFrame(
        index=source_datasets,
        columns=datasets,
        dtype=float,
    )
    acc_table = pd.DataFrame(
        index=source_datasets,
        columns=datasets,
        dtype=float,
    )

    print("\n=== Training hybrid model and running cross-domain evaluations ===")
    for src_name in source_datasets:
        print(f"\n=== Training hybrid model on source dataset: {src_name} ===")
        results = train_one_source(
            src_name=src_name,
            all_data=all_data,
            device=device,
            epochs=epochs,
            lr=1e-3,
            weight_decay=1e-4,
            gcn_hidden=256,
            fusion_hidden=256,
            dropout=dropout,
        )

        for tgt_name in datasets:
            macro_f1_table.loc[src_name, tgt_name] = results[tgt_name]["macro_f1"]
            acc_table.loc[src_name, tgt_name] = results[tgt_name]["acc"]

    print("\n=== Cross-domain Macro-F1 table (Train on row, Test on column) ===")
    print(macro_f1_table)

    print("\n=== Cross-domain Accuracy table (Train on row, Test on column) ===")
    print(acc_table)

    out_dir = os.path.join(base_dir, "cross_domain_results")
    os.makedirs(out_dir, exist_ok=True)

    macro_f1_path = os.path.join(out_dir, "hybrid_cross_domain_macro_f1_v3.csv")
    acc_path = os.path.join(out_dir, "hybrid_cross_domain_accuracy_v3.csv")

    macro_f1_table.to_csv(macro_f1_path)
    acc_table.to_csv(acc_path)

    print(f"\nSaved Macro-F1 table to: {macro_f1_path}")
    print(f"Saved Accuracy table to: {acc_path}")


# --------------------------------------------------------------
# Run: verification pass on just coaid + fakenewsnet first
# (once confirmed fixed, rerun with datasets_override=None for the full 4x4)
# --------------------------------------------------------------
run_cross_domain_hybrid(
    base_dir="drive/MyDrive/CSML/data",
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    batch_size=32,
    k=10,
    dropout=0.3,
    datasets_override=["coaid", "fakenewsnet"],
)
